In [139]:
from transformers import (
    RobertaTokenizer, 
    RobertaForMaskedLM, 
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    RobertaForSequenceClassification,
)
from datasets import load_dataset, Value
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, classification_report

### Pre-Training on Corpus

In [ ]:
# Loading model and tokenizer
roberta_model_mlm = RobertaForMaskedLM.from_pretrained('roberta-base')
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 995.24it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]               
RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
pretrained_dataset = load_dataset('avanishd/ground-news-2026') # Load dataset
print(pretrained_dataset)

DatasetDict({
    train: Dataset({
        features: ['outlet', 'bias', 'headline', 'summary', 'ground_news_interest_url'],
        num_rows: 4682
    })
    test: Dataset({
        features: ['outlet', 'bias', 'headline', 'summary', 'ground_news_interest_url'],
        num_rows: 1004
    })
    validation: Dataset({
        features: ['outlet', 'bias', 'headline', 'summary', 'ground_news_interest_url'],
        num_rows: 1003
    })
})


In [20]:
pretrained_dataset = pretrained_dataset.filter(lambda dataset: dataset['summary'] != 'null' and dataset['summary'] is not None and dataset['summary'] != '') # Filter out any null summaries

Filter: 100%|██████████| 980/980 [00:00<00:00, 37521.27 examples/s]


In [21]:
# Determines tokenization per data
def tokenize(data):
    return roberta_tokenizer(data['summary'], truncation=True, max_length=512, padding='max_length')

In [22]:
print(roberta_tokenizer("Monkey is a monkey but there is a rat rat", truncation=True, max_length=512, padding='max_length')) # Example output of tokenizer

{'input_ids': [0, 17312, 5282, 16, 10, 25684, 53, 89, 16, 10, 12378, 12378, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [23]:
tokenized_pretrained_dataset = pretrained_dataset.map(tokenize, batched=True, remove_columns=['summary']) # Tokenizes the descriptions for RoBERTa training

Map: 100%|██████████| 980/980 [00:00<00:00, 7380.67 examples/s]


In [26]:
data_collator = DataCollatorForLanguageModeling(tokenizer=roberta_tokenizer, mlm=True, mlm_probability=0.15) # Used to mask random tokens

training_args = TrainingArguments(
    output_dir='./domain-roberta-mlm',
    save_steps=10000,
    save_total_limit=2,
    logging_steps=500,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    fp16=True
) # Parameters used for trainer

trainer = Trainer(model=roberta_model_mlm,
                  args=training_args,
                  data_collator=data_collator,
                  train_dataset=tokenized_pretrained_dataset['train'])

In [27]:
trainer.train() # Training

Step,Training Loss
500,1.376999
1000,1.310041
1500,1.144443


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


TrainOutput(global_step=1722, training_loss=1.2546947359623395, metrics={'train_runtime': 1001.9235, 'train_samples_per_second': 13.75, 'train_steps_per_second': 1.719, 'total_flos': 3626745022365696.0, 'train_loss': 1.2546947359623395, 'epoch': 3.0})

In [28]:
# Saving model
roberta_model_mlm.save_pretrained('./domain-roberta-mlm')
roberta_tokenizer.save_pretrained('./domain-roberta-mlm')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.63it/s]


('./domain-roberta-mlm\\tokenizer_config.json',
 './domain-roberta-mlm\\tokenizer.json')

### Train for Classification

In [115]:
roberta_model_classifier = RobertaForSequenceClassification.from_pretrained('./domain-roberta-mlm', num_labels=7).to('cuda') # Load Pre-Trained model

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 691.23it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: ./domain-roberta-mlm
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [116]:
classifier_dataset = load_dataset('avanishd/ground-news-2026') # Load dataset

In [117]:
classifier_dataset = classifier_dataset.filter(lambda dataset: dataset['summary'] != 'null' and dataset['summary'] is not None and dataset['summary'] != '') # Filter out any null summaries
tokenized_classifier_dataset = classifier_dataset.map(tokenize, batched=True, remove_columns=['summary', 'outlet', 'headline', 'ground_news_interest_url']) # Tokenizes the descriptions for RoBERTa training
tokenized_classifier_dataset = tokenized_classifier_dataset.rename_column('bias', 'labels')

In [118]:
label_map = {"Far Left": 0, "Left": 1, "Lean Left": 2, "Center": 3, "Lean Right": 4, "Right": 5, "Far Right": 6}
tokenized_classifier_dataset = tokenized_classifier_dataset.map(lambda dataset: {'labels': int(label_map[dataset['labels']])})
tokenized_classifier_dataset = tokenized_classifier_dataset.cast_column('labels', Value('int32'))

In [119]:
print(tokenized_classifier_dataset['train']['labels'])
print(tokenized_classifier_dataset)

Column([3, 3, 3, 5, 2])
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 4592
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 976
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 980
    })
})


In [120]:
training_args = TrainingArguments(
    output_dir='./domain-roberta-classifier',
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=500,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    fp16=True
)  # Parameters used for trainer

trainer = Trainer(model=roberta_model_classifier,
                  args=training_args,
                  train_dataset=tokenized_classifier_dataset['train'],
                  eval_dataset=tokenized_classifier_dataset['validation'])

In [122]:
trainer.train() # Training

Epoch,Training Loss,Validation Loss
1,No log,1.320217
2,1.348164,1.213989
3,1.348164,1.194709
4,1.066839,1.213673
5,1.066839,1.237328


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=1435, training_loss=1.0892127272974739, metrics={'train_runtime': 657.1958, 'train_samples_per_second': 34.936, 'train_steps_per_second': 2.184, 'total_flos': 6041301030912000.0, 'train_loss': 1.0892127272974739, 'epoch': 5.0})

In [123]:
# Saving model
roberta_model_classifier.save_pretrained('./domain-roberta-classifier')
roberta_tokenizer.save_pretrained('./domain-roberta-classifier')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.73it/s]


('./domain-roberta-classifier\\tokenizer_config.json',
 './domain-roberta-classifier\\tokenizer.json')

In [137]:
predictions = trainer.predict(tokenized_classifier_dataset['test'])
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids


In [138]:
accuracy = accuracy_score(true_labels, preds)
mae = mean_absolute_error(true_labels, preds)
f1 = f1_score(true_labels, preds, average='macro')

print(f"Accuracy: {accuracy}")
print(f"MAE: {mae}")
print(f"Macro F1: {f1}")


Accuracy: 0.5030737704918032
MAE: 0.944672131147541
Macro F1: 0.37426266505615535


In [141]:
label_names = ["Left", "Lean Left", "Center", "Right", "Far Right"]
print(classification_report(true_labels, preds, target_names=label_names))

              precision    recall  f1-score   support

        Left       0.19      0.28      0.23        39
   Lean Left       0.43      0.44      0.43       246
      Center       0.55      0.71      0.62       376
       Right       0.56      0.37      0.45       275
   Far Right       1.00      0.07      0.14        40

    accuracy                           0.50       976
   macro avg       0.55      0.38      0.37       976
weighted avg       0.53      0.50      0.49       976

